In [ ]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolo26m.pt")  # load a pretrained model (recommended for training)

# Train the model
results = model.train(data="data.yaml", epochs=100, imgsz=640, batch=16)

New https://pypi.org/project/ultralytics/8.4.83 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.63 🚀 Python-3.12.3 torch-2.12.0+cu130 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48509MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name

In [1]:
!nvidia-smi

Tue Jun 30 12:10:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 6000 Ada Gene...    Off |   00000000:27:00.0 Off |                  Off |
| 30%   35C    P8             28W /  300W |    1102MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
%%writefile app_yolo.py
import streamlit as st
import cv2
import numpy as np
import os
from PIL import Image
from ultralytics import YOLO

st.set_page_config(layout="wide", page_title="AI Surveillance Platform - YOLOv11 Fixed")

st.title("🎯 Live Drone Surveillance Guardrails (Official VisDrone Model)")
st.sidebar.title("📊 Security Control Panel")

# Sahi VisDrone Validation Images Ka Path (Jo Sir ne download kiya)
IMAGE_DIR = "/home/intern/internship_projects/Umakanta/YOLO_CODE/datasets/VisDrone/VisDrone2019-DET-val/images"
if os.path.exists(IMAGE_DIR):
    img_list = [f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.png'))]
    selected_img = st.sidebar.selectbox("📂 Select Drone Surveillance Feed Image:", img_list if img_list else ["No Images Found"])
    img_path = os.path.join(IMAGE_DIR, selected_img)
else:
    st.error("VisDrone validation image folder not found!")
    st.stop()

state_key = f"poly_points_yolo_{selected_img}"
if state_key not in st.session_state:
    st.session_state[state_key] = []

if st.sidebar.button("🧹 Clear Drawn Alert Zone Points"):
    st.session_state[state_key] = []
    st.rerun()

@st.cache_resource
def load_custom_model():
    # PATH FIXED: Mapping to Sir's freshly trained yolo26m best weights!
    return YOLO('/home/intern/internship_projects/Ipseeta /AI SURVEILLANCE PLATFORM/visdrone-yolo/runs/detect/train-3/weights/best.pt')

model = load_custom_model()

def is_inside_polygon(x, y, poly):
    num = len(poly)
    j = num - 1
    c = False
    for i in range(num):
        if ((poly[i][1] > y) != (poly[j][1] > y)) and \
                (x < (poly[j][0] - poly[i][0]) * (y - poly[i][1]) / (poly[j][1] - poly[i][1] + 1e-6) + poly[i][0]):
            c = not c
        j = i
    return c

if img_path and os.path.exists(img_path):
    img_cv = cv2.imread(img_path)
    
    # Confidence threshold thoda optimal rakha hai drone objects ke liye
    results = model.predict(img_cv, conf=0.25, imgsz=640)[0]
    
    from streamlit_image_coordinates import streamlit_image_coordinates
    active_points = st.session_state[state_key]
    
    for pt in active_points:
        cv2.circle(img_cv, pt, 6, (0, 0, 255), -1)
        
    if len(active_points) > 1:
        pts = np.array(active_points, np.int32).reshape((-1, 1, 2))
        cv2.polylines(img_cv, [pts], False, (0, 0, 255), 2)
        
    violations_inside_zone = []
    zone_active = len(active_points) >= 3
    
    if zone_active:
        pts = np.array(active_points, np.int32).reshape((-1, 1, 2))
        cv2.polylines(img_cv, [pts], True, (0, 0, 255), 3)

    for idx, box in enumerate(results.boxes):
        coords = box.xyxy[0].tolist()
        x1, y1, x2, y2 = int(coords[0]), int(coords[1]), int(coords[2]), int(coords[3])
        
        cx = int((x1 + x2) / 2)
        cy = y2
        
        cls_id = int(box.cls[0])
        class_name = model.names[cls_id].capitalize()
        score = float(box.conf[0])
        
        breached = False
        if zone_active:
            breached = is_inside_polygon(cx, cy, active_points)
            
        if breached:
            color = (255, 0, 0)
            violations_inside_zone.append(f"🔴 {class_name} #{idx+1:02d} (Conf: {score:.2f})")
        else:
            color = (0, 255, 255)
            
        cv2.rectangle(img_cv, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img_cv, f"{class_name}", (x1, max(0, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 2)

    col1, col2 = st.columns([3, 1])
    
    with col1:
        st.subheader("🖱️ Click on image to draw security polygon area:")
        img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
        img_pil = Image.fromarray(img_rgb)
        
        value = streamlit_image_coordinates(img_pil, key="yolo_sir_data_final")
        if value is not None:
            new_point = (value["x"], value["y"])
            if new_point not in active_points:
                st.session_state[state_key].append(new_point)
                st.rerun()
                
    with col2:
        st.sidebar.metric(label="🚨 Violations in Clicked Area", value=len(violations_inside_zone))
        st.sidebar.subheader("📋 Active Breach Levels:")
        if len(active_points) < 3:
            st.sidebar.info("💡 Click at least 3 points on the image to activate the alert zone circuit.")
        elif violations_inside_zone:
            for v_name in violations_inside_zone:
                st.sidebar.error(v_name)
        else:
            st.sidebar.success("✅ Area Secure. Zero Breaches.")

Writing app_yolo.py
